In [0]:
import time
import random
import json
import threading
from datetime import datetime, timedelta

# Make students complete these paths
volume_path = "/Volumes/25280091-pa2/bronze/temp" 
flight_landing_zone = f"{volume_path}/flight_landing_zone_part2"
cdc_landing_zone = f"{volume_path}/cdc_flight_landing_zone_part2"

dbutils.fs.mkdirs(flight_landing_zone)
dbutils.fs.mkdirs(cdc_landing_zone)

# === DO NOT CHANGE ===
IATA_codes = ["LHE", "DXB", "DOH", "RUH", "AUH", "KWI", "JED"]
distance_map = {
    'LHE-DXB': 1233, 'LHE-DOH': 1447, 'LHE-RUH': 1741, 'LHE-AUH': 1298, 'LHE-KWI': 1579, 'LHE-JED': 2269, 'DXB-LHE': 1233, 
    'DXB-DOH': 234, 'DXB-RUH': 542, 'DXB-AUH': 72, 'DXB-KWI': 530, 'DXB-JED': 1055, 'DOH-LHE': 1447, 'DOH-DXB': 234, 
    'DOH-RUH': 307, 'DOH-AUH': 199, 'DOH-KWI': 352, 'DOH-JED': 826, 'RUH-LHE': 1741, 'RUH-DXB': 542, 'RUH-DOH': 307, 
    'RUH-AUH': 500, 'RUH-KWI': 305, 'RUH-JED': 529, 'AUH-LHE': 1298, 'AUH-DXB': 72, 'AUH-DOH': 199, 'AUH-RUH': 500, 
    'AUH-KWI': 528, 'AUH-JED': 1002, 'KWI-LHE': 1579, 'KWI-DXB': 530, 'KWI-DOH': 352, 'KWI-RUH': 305, 'KWI-AUH': 528, 
    'KWI-JED': 757, 'JED-LHE': 2269, 'JED-DXB': 1055, 'JED-DOH': 826, 'JED-RUH': 529, 'JED-AUH': 1002, 'JED-KWI': 757
}
location_lookup = {
    "LHE": "Lahore, PB", "DXB": "Dubai, DU", "DOH": "Doha, QA",
    "RUH": "Riyadh, RI", "AUH": "Abu Dhabi, AZ", "KWI": "Kuwait City, KW",
    "JED": "Jeddah, MK"
}

In [0]:
# === DO NOT CHANGE ===
def generate_flight_data(seed=0):
    random.seed(seed)

    # File Structure
    rows = ["date,delay,distance,origin,destination"]

    # Random Date
    base_time = datetime(2025, random.randint(1, 12), random.randint(1, 28), random.randint(0, 23), random.randint(0, 59))
    
    # Random number of rows per file
    num_rows = random.randint(10, 50)
    
    # File Row Generator
    for row_idx in range(num_rows):
        origin = random.choice(IATA_codes)
        dest = random.choice([c for c in IATA_codes if c != origin])
        route = f"{origin}-{dest}"
        dist = distance_map.get(route, random.randint(300, 2000))
        current_row_time = base_time + timedelta(minutes=row_idx)
        rows.append(f"{current_row_time.strftime('%m%d%H%M')},{random.randint(-15, 300)},{dist},{origin},{dest}")

    return "\n".join(rows)

# === DO NOT CHANGE ===
def generate_cdc_updates(seed=0, seed_hist=0):
    random.seed(seed)
    data_pool = []

    # Random Date
    base_time = datetime(2025, random.randint(1, 12), random.randint(1, 28), random.randint(0, 23), random.randint(0, 59))

    # Random number of rows per file
    num_rows = random.randint(10, 50)
    
    # File Row Generator
    for row_idx in range(num_rows):
        origin = random.choice(IATA_codes)
        dest = random.choice([c for c in IATA_codes if c != origin])
        standardized_route = "-".join(sorted([origin, dest]))
        
        action = "DELETE" if random.random() < 0.10 else "UPDATE"
        seq = 3
        
        if seed > 0 and seed == seed_hist:
            seq += random.randint(1, 3) if random.randint(1, 10) <= 8 else random.randint(-2, -1)
            
        event = {
            "update_ts": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "action": action,
            "seq": seq,
            "payload": {
                "date": (base_time + timedelta(minutes=row_idx)).strftime("%m%d%H%M"),
                "departure": location_lookup.get(origin),
                "arrival": location_lookup.get(dest),
                "standardized_route": standardized_route,
                "distance": distance_map.get(standardized_route, random.randint(300, 2000)),
                "delay": random.randint(-15, 300)
            }
        }
        data_pool.append(event)

    random.seed(seed)
    random.shuffle(data_pool)
    return "\n".join([json.dumps(record) for record in data_pool])

In [0]:
import time

num_iterations = 5
interval = 5
seed_hist = -1

volume_path = "/Volumes/25280091-pa2/bronze/temp" 

for i in range(num_iterations):
    flight_content = generate_flight_data(seed=i)
    flight_path = f"{volume_path}/flight_landing_zone_part2/batch_{i}.csv"
    dbutils.fs.put(flight_path, flight_content, overwrite=True)

    cdc_content = generate_cdc_updates(seed=i, seed_hist=seed_hist)
    cdc_path = f"{volume_path}/cdc_zone_part2/cdc_batch_{i}.json"
    dbutils.fs.put(cdc_path, cdc_content, overwrite=True)

    seed_hist = i
    print(f"Generated matching batch {i} at {datetime.now().strftime('%H:%M:%S')}")
    time.sleep(interval)

Wrote 689 bytes.
Wrote 5666 bytes.
Generated matching batch 0 at 20:58:35
Wrote 463 bytes.
Wrote 3717 bytes.
Generated matching batch 1 at 20:58:40
Wrote 534 bytes.
Wrote 4350 bytes.
Generated matching batch 2 at 20:58:46
Wrote 852 bytes.
Wrote 7187 bytes.
Generated matching batch 3 at 20:58:52
Wrote 909 bytes.
Wrote 7574 bytes.
Generated matching batch 4 at 20:58:58


In [0]:
display(dbutils.fs.ls("/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2"))
display(dbutils.fs.ls("/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2"))

path,name,size,modificationTime
dbfs:/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_0.csv,batch_0.csv,689,1774731515000
dbfs:/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_1.csv,batch_1.csv,463,1774731521000
dbfs:/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_2.csv,batch_2.csv,534,1774731527000
dbfs:/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_3.csv,batch_3.csv,852,1774731532000
dbfs:/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_4.csv,batch_4.csv,909,1774731538000


path,name,size,modificationTime
dbfs:/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_0.json,cdc_batch_0.json,5666,1774731515000
dbfs:/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_1.json,cdc_batch_1.json,3717,1774731521000
dbfs:/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_2.json,cdc_batch_2.json,4350,1774731527000
dbfs:/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_3.json,cdc_batch_3.json,7187,1774731533000
dbfs:/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_4.json,cdc_batch_4.json,7574,1774731538000


In [0]:
display(
    spark.read
         .option("header", "true")
         .csv("/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_0.csv")
)

date,delay,distance,origin,destination
07251302,140,500,AUH,RUH
07251303,96,307,RUH,DOH
07251304,56,72,AUH,DXB
07251305,113,2269,JED,LHE
07251306,293,1002,AUH,JED
07251307,22,542,DXB,RUH
07251308,226,757,JED,KWI
07251309,207,1298,AUH,LHE
07251310,89,352,DOH,KWI
07251311,251,500,AUH,RUH


In [0]:
display(
    spark.read
         .json("/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_0.json")
)

action,payload,seq,update_ts
UPDATE,"List(Doha, QA, 07251316, 89, Abu Dhabi, AZ, 199, AUH-DOH)",3,2026-03-28 20:58:34
UPDATE,"List(Riyadh, RI, 07251302, 140, Abu Dhabi, AZ, 500, AUH-RUH)",3,2026-03-28 20:58:34
UPDATE,"List(Riyadh, RI, 07251325, 237, Jeddah, MK, 529, JED-RUH)",3,2026-03-28 20:58:34
UPDATE,"List(Kuwait City, KW, 07251320, 228, Lahore, PB, 1579, KWI-LHE)",3,2026-03-28 20:58:34
UPDATE,"List(Jeddah, MK, 07251308, 229, Abu Dhabi, AZ, 1002, AUH-JED)",3,2026-03-28 20:58:34
UPDATE,"List(Kuwait City, KW, 07251307, 146, Riyadh, RI, 305, KWI-RUH)",3,2026-03-28 20:58:34
UPDATE,"List(Kuwait City, KW, 07251309, 265, Riyadh, RI, 305, KWI-RUH)",3,2026-03-28 20:58:34
UPDATE,"List(Doha, QA, 07251312, 82, Kuwait City, KW, 352, DOH-KWI)",3,2026-03-28 20:58:34
UPDATE,"List(Dubai, DU, 07251324, 286, Abu Dhabi, AZ, 72, AUH-DXB)",3,2026-03-28 20:58:34
UPDATE,"List(Dubai, DU, 07251322, 261, Lahore, PB, 1233, DXB-LHE)",3,2026-03-28 20:58:34


In [0]:
print(dbutils.fs.head("/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_0.csv", 1000))

date,delay,distance,origin,destination
07251302,140,500,AUH,RUH
07251303,96,307,RUH,DOH
07251304,56,72,AUH,DXB
07251305,113,2269,JED,LHE
07251306,293,1002,AUH,JED
07251307,22,542,DXB,RUH
07251308,226,757,JED,KWI
07251309,207,1298,AUH,LHE
07251310,89,352,DOH,KWI
07251311,251,500,AUH,RUH
07251312,265,1447,DOH,LHE
07251313,189,1233,LHE,DXB
07251314,-15,757,KWI,JED
07251315,155,500,AUH,RUH
07251316,17,1055,DXB,JED
07251317,107,530,DXB,KWI
07251318,263,1055,JED,DXB
07251319,148,1741,RUH,LHE
07251320,139,500,AUH,RUH
07251321,48,199,AUH,DOH
07251322,261,199,AUH,DOH
07251323,285,530,DXB,KWI
07251324,290,199,DOH,AUH
07251325,279,529,JED,RUH
07251326,81,542,DXB,RUH
07251327,298,1055,JED,DXB


In [0]:
print(dbutils.fs.head("/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_0.json", 2000))

[Truncated to first 2000 bytes]
{"update_ts": "2026-03-28 20:58:34", "action": "UPDATE", "seq": 3, "payload": {"date": "07251316", "departure": "Abu Dhabi, AZ", "arrival": "Doha, QA", "standardized_route": "AUH-DOH", "distance": 199, "delay": 89}}
{"update_ts": "2026-03-28 20:58:34", "action": "UPDATE", "seq": 3, "payload": {"date": "07251302", "departure": "Abu Dhabi, AZ", "arrival": "Riyadh, RI", "standardized_route": "AUH-RUH", "distance": 500, "delay": 140}}
{"update_ts": "2026-03-28 20:58:34", "action": "UPDATE", "seq": 3, "payload": {"date": "07251325", "departure": "Jeddah, MK", "arrival": "Riyadh, RI", "standardized_route": "JED-RUH", "distance": 529, "delay": 237}}
{"update_ts": "2026-03-28 20:58:34", "action": "UPDATE", "seq": 3, "payload": {"date": "07251320", "departure": "Lahore, PB", "arrival": "Kuwait City, KW", "standardized_route": "KWI-LHE", "distance": 1579, "delay": 228}}
{"update_ts": "2026-03-28 20:58:34", "action": "UPDATE", "seq": 3, "payload": {"date": "0725130

In [0]:
flight_files = dbutils.fs.ls("/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2")
cdc_files = dbutils.fs.ls("/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2")

print("Flight files:", len(flight_files))
print("CDC files:", len(cdc_files))

Flight files: 5
CDC files: 5


In [0]:
print("=== Flight files ===")
for f in dbutils.fs.ls("/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2"):
    print(f.name)

print("\n=== CDC files ===")
for f in dbutils.fs.ls("/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2"):
    print(f.name)

print("\n=== Sample flight file ===")
print(dbutils.fs.head("/Volumes/25280091-pa2/bronze/temp/flight_landing_zone_part2/batch_0.csv", 500))

print("\n=== Sample CDC file ===")
print(dbutils.fs.head("/Volumes/25280091-pa2/bronze/temp/cdc_zone_part2/cdc_batch_0.json", 1000))

=== Flight files ===
batch_0.csv
batch_1.csv
batch_2.csv
batch_3.csv
batch_4.csv

=== CDC files ===
cdc_batch_0.json
cdc_batch_1.json
cdc_batch_2.json
cdc_batch_3.json
cdc_batch_4.json

=== Sample flight file ===
[Truncated to first 500 bytes]
date,delay,distance,origin,destination
07251302,140,500,AUH,RUH
07251303,96,307,RUH,DOH
07251304,56,72,AUH,DXB
07251305,113,2269,JED,LHE
07251306,293,1002,AUH,JED
07251307,22,542,DXB,RUH
07251308,226,757,JED,KWI
07251309,207,1298,AUH,LHE
07251310,89,352,DOH,KWI
07251311,251,500,AUH,RUH
07251312,265,1447,DOH,LHE
07251313,189,1233,LHE,DXB
07251314,-15,757,KWI,JED
07251315,155,500,AUH,RUH
07251316,17,1055,DXB,JED
07251317,107,530,DXB,KWI
07251318,263,1055,JED,DXB
07251319,148,1741,RUH,LHE
07251320,

=== Sample CDC file ===
[Truncated to first 1000 bytes]
{"update_ts": "2026-03-28 20:58:34", "action": "UPDATE", "seq": 3, "payload": {"date": "07251316", "departure": "Abu Dhabi, AZ", "arrival": "Doha, QA", "standardized_route": "AUH-DOH", "distance": 1